In [ ]:
import os
from pathlib import Path

REVIEW_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "REVIEW_CODE_ROOT").is_file()),
    None,
)
if REVIEW_ROOT is None:
    raise RuntimeError("Open this notebook from the extracted review package.")
os.chdir(REVIEW_ROOT)
REVIEW_FREQUENCY = os.environ.get("BFORTPROB_FREQUENCY", "daily")
if REVIEW_FREQUENCY not in {"daily", "weekly"}:
    raise ValueError("BFORTPROB_FREQUENCY must be daily or weekly.")


Generate:

- quantile reliability for 0.1–0.9
- 80% interval coverage using q0.1 and q0.9 interval width
- downside exceedance at h=1,5,20
- daily and weekly summaries

# Daily

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# CONFIG
# ============================================================

LOOKBACK = 250
HORIZON = 20

QUANTILE_LEVELS_EXPECTED = np.arange(
    0.1,
    1.0,
    0.1,
)

SELECTED_HORIZONS = [
    1,
    5,
    20,
]

ASSET_ORDER = {
    "G7": "FX",
    "EM": "EM FX",
    "MSCI": "MSCI",
    "Rates": "Rates",
    "REIT": "REIT",
    "Equities": "Equities",
    "ETF": "ETF",
    "Crypto": "Crypto",
}


# ------------------------------------------------------------
# DAILY
# ------------------------------------------------------------

BASE_DIR = Path(
    r"."
    r"\BForTFinProbability\BForTFin_results"
    r"\standardised\output_daily"
)

DATA_DIR = Path(
    r"."
    r"\BForTFinProbability\BForTFin_log_return"
    r"\arima_split_datasets\output_daily"
)


MODEL_DIRS = {
    "GARCH(1,1)": BASE_DIR / "garch_standardized_train_test",
    "LightGBM": BASE_DIR / "lightgbm_standardized_train_test",
    "Sundial": BASE_DIR / "sundial_standardized_train_test",
    "Lag-Llama": BASE_DIR / "lag-llama_standardized_train_test",
    "Chronos-T5": BASE_DIR / "chronos_standardized_train_test",
    "Toto": BASE_DIR / "toto_standardized_train_test",
    "TimesFM": BASE_DIR / "timesfm_standardized_train_test",
    "Moirai": BASE_DIR / "moirai_standardized_train_test",
    "Moirai-MoE": BASE_DIR / "moirai_moe_standardized_train_test",
}


MODEL_SUFFIXES = {
    "GARCH(1,1)": "_garch_1_1",
    "LightGBM": "_lightgbm",
    "Sundial": "_sundial",
    "Lag-Llama": "_lag-llama",
    "Chronos-T5": "_chronos-t5-small",
    "Toto": "_toto",
    "TimesFM": "_timesfm-2.5-200m",
    "Moirai": "_moirai-2.0-r-small",
    "Moirai-MoE": "_moirai-moe-1.0-r-base",
}


# ============================================================
# LOAD TEST TARGETS
# ============================================================

def load_test_y(test_path):

    df = pd.read_csv(test_path)

    y_cols = [
        f"Y_{i}"
        for i in range(HORIZON)
    ]

    Y = df[y_cols].to_numpy(
        dtype=float
    )

    return Y


# ============================================================
# GARCH FILENAME HELPERS
# ============================================================

def clean_garch_dataset_name(dataset_name):

    return (
        dataset_name
        .lower()
        .replace(" historical data", "")
        .replace(
            "_no_preprocess_lookback-250_horizon-20_slide-1",
            "",
        )
        .replace(" ", "_")
        .replace(".", "")
    )


def find_garch_quantile_file(
    model_root,
    asset_class,
    dataset_name,
):

    asset_dir = model_root / asset_class

    clean_name = clean_garch_dataset_name(
        dataset_name
    )

    prefix = (
        f"{asset_class.lower()}_"
        f"{clean_name}_"
        f"garch_1_1_train_test"
    )

    expected = (
        asset_dir
        / f"{prefix}_quantiles.npz"
    )

    if expected.exists():
        return expected

    matches = list(
        asset_dir.glob(
            f"*{clean_name}*"
            f"garch_1_1_train_test*"
            f"quantiles.npz"
        )
    )

    if len(matches) == 1:
        return matches[0]

    if len(matches) == 0:
        raise FileNotFoundError(
            f"No GARCH quantile file found for "
            f"{dataset_name} in {asset_class}"
        )

    raise RuntimeError(
        f"Multiple GARCH quantile files found for "
        f"{dataset_name}: {matches}"
    )


# ============================================================
# FIND MODEL QUANTILE FILE
# ============================================================

def find_quantile_file(
    model_name,
    asset_class,
    dataset_name,
):

    model_root = MODEL_DIRS[
        model_name
    ]

    if model_name == "GARCH(1,1)":

        return find_garch_quantile_file(
            model_root=model_root,
            asset_class=asset_class,
            dataset_name=dataset_name,
        )

    suffix = MODEL_SUFFIXES[
        model_name
    ]

    model_asset_dir = (
        model_root
        / asset_class
    )

    matches = list(
        model_asset_dir.glob(
            f"test_{dataset_name}"
            f"{suffix}_quantiles.npz"
        )
    )

    if len(matches) == 1:
        return matches[0]

    raise FileNotFoundError(
        f"Could not uniquely identify quantiles "
        f"for {model_name}, {asset_class}, "
        f"{dataset_name}. "
        f"Found {len(matches)} matches."
    )


# ============================================================
# LOAD QUANTILES
# ============================================================

def load_quantiles(path):

    with np.load(path) as q_file:

        quantiles = np.asarray(
            q_file["quantiles"],
            dtype=float,
        )

        levels = np.asarray(
            q_file["quantile_levels"],
            dtype=float,
        )

    return quantiles, levels


# ============================================================
# QUANTILE INDEX HELPER
# ============================================================

def get_quantile_index(
    levels,
    tau,
):

    idx = np.argmin(
        np.abs(levels - tau)
    )

    if not np.isclose(
        levels[idx],
        tau,
    ):
        raise ValueError(
            f"No quantile tau={tau} found. "
            f"Available levels: {levels}"
        )

    return idx


# ============================================================
# CHECK QUANTILE SHAPE / ORDER
# ============================================================

def validate_quantiles(
    Y,
    Q,
    levels,
):

    if Q.ndim != 3:
        raise ValueError(
            f"Expected quantiles with 3 dimensions "
            f"(origins, horizons, quantiles), "
            f"got shape {Q.shape}"
        )

    if Q.shape[0] != Y.shape[0]:
        raise ValueError(
            f"Origin mismatch: "
            f"Y={Y.shape}, Q={Q.shape}"
        )

    if Q.shape[1] != Y.shape[1]:
        raise ValueError(
            f"Horizon mismatch: "
            f"Y={Y.shape}, Q={Q.shape}"
        )

    if Q.shape[2] != len(levels):
        raise ValueError(
            f"Quantile dimension mismatch: "
            f"Q={Q.shape}, levels={len(levels)}"
        )

    # Check that reported levels increase
    if not np.all(
        np.diff(levels) > 0
    ):
        raise ValueError(
            f"Quantile levels not strictly increasing: "
            f"{levels}"
        )


# ============================================================
# 1. QUANTILE RELIABILITY
# ============================================================

def calculate_reliability(
    Y,
    Q,
    levels,
):
    """
    For each predicted quantile tau:

        observed frequency =
            mean(Y <= q_tau)

    Perfect calibration implies:

        observed frequency ~= tau
    """

    rows = []

    for j, tau in enumerate(levels):

        q_tau = Q[:, :, j]

        observed_frequency = float(
            np.mean(
                Y <= q_tau
            )
        )

        calibration_error = (
            observed_frequency
            - float(tau)
        )

        rows.append({
            "quantile_level": float(tau),
            "observed_frequency": (
                observed_frequency
            ),
            "calibration_error": (
                calibration_error
            ),
            "absolute_calibration_error": (
                abs(calibration_error)
            ),
        })

    return rows


# ============================================================
# 2. 80% INTERVAL COVERAGE + WIDTH
# ============================================================

def calculate_interval_80(
    Y,
    Q,
    levels,
):
    """
    Nominal 80% interval:

        [q_0.1, q_0.9]

    Perfect empirical coverage would be ~0.80.
    """

    q10_idx = get_quantile_index(
        levels,
        0.1,
    )

    q90_idx = get_quantile_index(
        levels,
        0.9,
    )

    lower = Q[:, :, q10_idx]
    upper = Q[:, :, q90_idx]

    coverage = float(
        np.mean(
            (Y >= lower)
            & (Y <= upper)
        )
    )

    width = float(
        np.mean(
            upper - lower
        )
    )

    return {
        "coverage_80": coverage,
        "coverage_error": (
            coverage - 0.80
        ),
        "absolute_coverage_error": (
            abs(coverage - 0.80)
        ),
        "mean_interval_width": width,
    }


# ============================================================
# 3. DOWNSIDE EXCEEDANCE AT h = 1, 5, 20
# ============================================================

def calculate_downside_exceedance(
    Y,
    Q,
    levels,
):
    """
    Frequency with which the realized return falls
    below the predicted 10% quantile.

    For a calibrated q_0.1 forecast this should be
    approximately 0.10.
    """

    q10_idx = get_quantile_index(
        levels,
        0.1,
    )

    q10 = Q[:, :, q10_idx]

    rows = []

    for h in SELECTED_HORIZONS:

        idx = h - 1

        observed_frequency = float(
            np.mean(
                Y[:, idx]
                <= q10[:, idx]
            )
        )

        rows.append({
            "horizon": h,
            "nominal_probability": 0.1,
            "observed_frequency": (
                observed_frequency
            ),
            "calibration_error": (
                observed_frequency
                - 0.1
            ),
            "absolute_calibration_error": (
                abs(
                    observed_frequency
                    - 0.1
                )
            ),
        })

    return rows


# ============================================================
# RUN ALL DAILY DATASETS
# ============================================================

reliability_rows = []
interval_rows = []
downside_rows = []


for model_name in MODEL_DIRS:

    print()
    print("=" * 70)
    print(f"MODEL: {model_name}")
    print("=" * 70)

    for asset_class, asset_name in ASSET_ORDER.items():

        test_asset_dir = (
            DATA_DIR
            / asset_class
        )

        test_files = sorted(
            test_asset_dir.glob(
                "test_*.csv"
            )
        )

        for test_path in test_files:

            try:

                dataset_name = (
                    test_path.stem
                    .removeprefix("test_")
                )

                # --------------------------------------------
                # Actual test targets
                # --------------------------------------------

                Y = load_test_y(
                    test_path
                )

                # --------------------------------------------
                # Saved model quantiles
                # --------------------------------------------

                quantile_path = (
                    find_quantile_file(
                        model_name=model_name,
                        asset_class=asset_class,
                        dataset_name=dataset_name,
                    )
                )

                Q, levels = (
                    load_quantiles(
                        quantile_path
                    )
                )

                validate_quantiles(
                    Y=Y,
                    Q=Q,
                    levels=levels,
                )

                # --------------------------------------------
                # Reliability
                # --------------------------------------------

                reliability = (
                    calculate_reliability(
                        Y=Y,
                        Q=Q,
                        levels=levels,
                    )
                )

                for row in reliability:

                    reliability_rows.append({
                        "model": model_name,
                        "asset_class": asset_name,
                        "dataset": dataset_name,
                        **row,
                    })

                # --------------------------------------------
                # 80% interval
                # --------------------------------------------

                interval = (
                    calculate_interval_80(
                        Y=Y,
                        Q=Q,
                        levels=levels,
                    )
                )

                interval_rows.append({
                    "model": model_name,
                    "asset_class": asset_name,
                    "dataset": dataset_name,
                    **interval,
                })

                # --------------------------------------------
                # Downside exceedance
                # --------------------------------------------

                downside = (
                    calculate_downside_exceedance(
                        Y=Y,
                        Q=Q,
                        levels=levels,
                    )
                )

                for row in downside:

                    downside_rows.append({
                        "model": model_name,
                        "asset_class": asset_name,
                        "dataset": dataset_name,
                        **row,
                    })

                print(
                    f"  {dataset_name}: done"
                )

            except Exception as exc:

                print(
                    f"  ERROR {test_path.name}: "
                    f"{exc}"
                )


# ============================================================
# DATAFRAMES
# ============================================================

reliability_df = pd.DataFrame(
    reliability_rows
)

interval_df = pd.DataFrame(
    interval_rows
)

downside_df = pd.DataFrame(
    downside_rows
)


# ============================================================
# MODEL-LEVEL SUMMARY
#
# Equal-series weighting:
# each dataset contributes equally.
# ============================================================

reliability_summary = (
    reliability_df
    .groupby(
        [
            "model",
            "quantile_level",
        ],
        as_index=False,
    )
    .agg(
        mean_observed_frequency=(
            "observed_frequency",
            "mean",
        ),
        mean_calibration_error=(
            "calibration_error",
            "mean",
        ),
        mean_absolute_calibration_error=(
            "absolute_calibration_error",
            "mean",
        ),
        n_datasets=(
            "dataset",
            "count",
        ),
    )
)


interval_summary = (
    interval_df
    .groupby(
        "model",
        as_index=False,
    )
    .agg(
        mean_coverage_80=(
            "coverage_80",
            "mean",
        ),
        mean_absolute_coverage_error=(
            "absolute_coverage_error",
            "mean",
        ),
        median_coverage_80=(
            "coverage_80",
            "median",
        ),
        mean_interval_width=(
            "mean_interval_width",
            "mean",
        ),
        median_interval_width=(
            "mean_interval_width",
            "median",
        ),
        n_datasets=(
            "dataset",
            "count",
        ),
    )
)


downside_summary = (
    downside_df
    .groupby(
        [
            "model",
            "horizon",
        ],
        as_index=False,
    )
    .agg(
        mean_observed_frequency=(
            "observed_frequency",
            "mean",
        ),
        mean_calibration_error=(
            "calibration_error",
            "mean",
        ),
        mean_absolute_calibration_error=(
            "absolute_calibration_error",
            "mean",
        ),
        n_datasets=(
            "dataset",
            "count",
        ),
    )
)


# ============================================================
# ASSET-CLASS SUMMARY
# ============================================================

reliability_asset_summary = (
    reliability_df
    .groupby(
        [
            "model",
            "asset_class",
            "quantile_level",
        ],
        as_index=False,
    )
    .agg(
        mean_observed_frequency=(
            "observed_frequency",
            "mean",
        ),
        mean_absolute_calibration_error=(
            "absolute_calibration_error",
            "mean",
        ),
        n_datasets=(
            "dataset",
            "count",
        ),
    )
)


interval_asset_summary = (
    interval_df
    .groupby(
        [
            "model",
            "asset_class",
        ],
        as_index=False,
    )
    .agg(
        mean_coverage_80=(
            "coverage_80",
            "mean",
        ),
        mean_absolute_coverage_error=(
            "absolute_coverage_error",
            "mean",
        ),
        mean_interval_width=(
            "mean_interval_width",
            "mean",
        ),
        n_datasets=(
            "dataset",
            "count",
        ),
    )
)


downside_asset_summary = (
    downside_df
    .groupby(
        [
            "model",
            "asset_class",
            "horizon",
        ],
        as_index=False,
    )
    .agg(
        mean_observed_frequency=(
            "observed_frequency",
            "mean",
        ),
        mean_absolute_calibration_error=(
            "absolute_calibration_error",
            "mean",
        ),
        n_datasets=(
            "dataset",
            "count",
        ),
    )
)


# ============================================================
# SAVE
# ============================================================

reliability_df.to_csv(
    BASE_DIR
    / "calibration_reliability_daily.csv",
    index=False,
)

interval_df.to_csv(
    BASE_DIR
    / "calibration_interval80_daily.csv",
    index=False,
)

downside_df.to_csv(
    BASE_DIR
    / "calibration_downside_horizons_daily.csv",
    index=False,
)


reliability_summary.to_csv(
    BASE_DIR
    / "calibration_reliability_summary_daily.csv",
    index=False,
)

interval_summary.to_csv(
    BASE_DIR
    / "calibration_interval80_summary_daily.csv",
    index=False,
)

downside_summary.to_csv(
    BASE_DIR
    / "calibration_downside_summary_daily.csv",
    index=False,
)


reliability_asset_summary.to_csv(
    BASE_DIR
    / "calibration_reliability_asset_class_daily.csv",
    index=False,
)

interval_asset_summary.to_csv(
    BASE_DIR
    / "calibration_interval80_asset_class_daily.csv",
    index=False,
)

downside_asset_summary.to_csv(
    BASE_DIR
    / "calibration_downside_asset_class_daily.csv",
    index=False,
)


print()
print("=" * 70)
print("SAVED DAILY CALIBRATION DIAGNOSTICS")
print("=" * 70)

# Weekly

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# CONFIG
# ============================================================

LOOKBACK = 250
HORIZON = 20

QUANTILE_LEVELS_EXPECTED = np.arange(
    0.1,
    1.0,
    0.1,
)

SELECTED_HORIZONS = [
    1,
    5,
    20,
]

ASSET_ORDER = {
    "G7": "FX",
    "EM": "EM FX",
    "MSCI": "MSCI",
    "Rates": "Rates",
    "REIT": "REIT",
    "Equities": "Equities",
    "ETF": "ETF",
    "Crypto": "Crypto",
}


# ------------------------------------------------------------
# DAILY
# ------------------------------------------------------------

BASE_DIR = Path(
    r"."
    r"\BForTFinProbability\BForTFin_results"
    r"\standardised\output_weekly"
)

DATA_DIR = Path(
    r"."
    r"\BForTFinProbability\BForTFin_log_return"
    r"\arima_split_datasets\output_weekly"
)


MODEL_DIRS = {
    "GARCH(1,1)": BASE_DIR / "garch_standardized_train_test",
    "LightGBM": BASE_DIR / "lightgbm_standardized_train_test",
    "Sundial": BASE_DIR / "sundial_standardized_train_test",
    "Lag-Llama": BASE_DIR / "lag-llama_standardized_train_test",
    "Chronos-T5": BASE_DIR / "chronos_standardized_train_test",
    "Toto": BASE_DIR / "toto_standardized_train_test",
    "TimesFM": BASE_DIR / "timesfm_standardized_train_test",
    "Moirai": BASE_DIR / "moirai_standardized_train_test",
    "Moirai-MoE": BASE_DIR / "moirai_moe_standardized_train_test",
}


MODEL_SUFFIXES = {
    "GARCH(1,1)": "_garch_1_1",
    "LightGBM": "_lightgbm",
    "Sundial": "_sundial",
    "Lag-Llama": "_lag-llama",
    "Chronos-T5": "_chronos-t5-small",
    "Toto": "_toto",
    "TimesFM": "_timesfm-2.5-200m",
    "Moirai": "_moirai-2.0-r-small",
    "Moirai-MoE": "_moirai-moe-1.0-r-base",
}


# ============================================================
# LOAD TEST TARGETS
# ============================================================

def load_test_y(test_path):

    df = pd.read_csv(test_path)

    y_cols = [
        f"Y_{i}"
        for i in range(HORIZON)
    ]

    Y = df[y_cols].to_numpy(
        dtype=float
    )

    return Y


# ============================================================
# GARCH FILENAME HELPERS
# ============================================================

def clean_garch_dataset_name(dataset_name):

    return (
        dataset_name
        .lower()
        .replace(" historical data", "")
        .replace(
            "_no_preprocess_lookback-250_horizon-20_slide-1",
            "",
        )
        .replace(" ", "_")
        .replace(".", "")
    )


def find_garch_quantile_file(
    model_root,
    asset_class,
    dataset_name,
):

    asset_dir = model_root / asset_class

    clean_name = clean_garch_dataset_name(
        dataset_name
    )

    prefix = (
        f"{asset_class.lower()}_"
        f"{clean_name}_"
        f"garch_1_1_train_test"
    )

    expected = (
        asset_dir
        / f"{prefix}_quantiles.npz"
    )

    if expected.exists():
        return expected

    matches = list(
        asset_dir.glob(
            f"*{clean_name}*"
            f"garch_1_1_train_test*"
            f"quantiles.npz"
        )
    )

    if len(matches) == 1:
        return matches[0]

    if len(matches) == 0:
        raise FileNotFoundError(
            f"No GARCH quantile file found for "
            f"{dataset_name} in {asset_class}"
        )

    raise RuntimeError(
        f"Multiple GARCH quantile files found for "
        f"{dataset_name}: {matches}"
    )


# ============================================================
# FIND MODEL QUANTILE FILE
# ============================================================

def find_quantile_file(
    model_name,
    asset_class,
    dataset_name,
):

    model_root = MODEL_DIRS[
        model_name
    ]

    if model_name == "GARCH(1,1)":

        return find_garch_quantile_file(
            model_root=model_root,
            asset_class=asset_class,
            dataset_name=dataset_name,
        )

    suffix = MODEL_SUFFIXES[
        model_name
    ]

    model_asset_dir = (
        model_root
        / asset_class
    )

    matches = list(
        model_asset_dir.glob(
            f"test_{dataset_name}"
            f"{suffix}_quantiles.npz"
        )
    )

    if len(matches) == 1:
        return matches[0]

    raise FileNotFoundError(
        f"Could not uniquely identify quantiles "
        f"for {model_name}, {asset_class}, "
        f"{dataset_name}. "
        f"Found {len(matches)} matches."
    )


# ============================================================
# LOAD QUANTILES
# ============================================================

def load_quantiles(path):

    with np.load(path) as q_file:

        quantiles = np.asarray(
            q_file["quantiles"],
            dtype=float,
        )

        levels = np.asarray(
            q_file["quantile_levels"],
            dtype=float,
        )

    return quantiles, levels


# ============================================================
# QUANTILE INDEX HELPER
# ============================================================

def get_quantile_index(
    levels,
    tau,
):

    idx = np.argmin(
        np.abs(levels - tau)
    )

    if not np.isclose(
        levels[idx],
        tau,
    ):
        raise ValueError(
            f"No quantile tau={tau} found. "
            f"Available levels: {levels}"
        )

    return idx


# ============================================================
# CHECK QUANTILE SHAPE / ORDER
# ============================================================

def validate_quantiles(
    Y,
    Q,
    levels,
):

    if Q.ndim != 3:
        raise ValueError(
            f"Expected quantiles with 3 dimensions "
            f"(origins, horizons, quantiles), "
            f"got shape {Q.shape}"
        )

    if Q.shape[0] != Y.shape[0]:
        raise ValueError(
            f"Origin mismatch: "
            f"Y={Y.shape}, Q={Q.shape}"
        )

    if Q.shape[1] != Y.shape[1]:
        raise ValueError(
            f"Horizon mismatch: "
            f"Y={Y.shape}, Q={Q.shape}"
        )

    if Q.shape[2] != len(levels):
        raise ValueError(
            f"Quantile dimension mismatch: "
            f"Q={Q.shape}, levels={len(levels)}"
        )

    # Check that reported levels increase
    if not np.all(
        np.diff(levels) > 0
    ):
        raise ValueError(
            f"Quantile levels not strictly increasing: "
            f"{levels}"
        )


# ============================================================
# 1. QUANTILE RELIABILITY
# ============================================================

def calculate_reliability(
    Y,
    Q,
    levels,
):
    """
    For each predicted quantile tau:

        observed frequency =
            mean(Y <= q_tau)

    Perfect calibration implies:

        observed frequency ~= tau
    """

    rows = []

    for j, tau in enumerate(levels):

        q_tau = Q[:, :, j]

        observed_frequency = float(
            np.mean(
                Y <= q_tau
            )
        )

        calibration_error = (
            observed_frequency
            - float(tau)
        )

        rows.append({
            "quantile_level": float(tau),
            "observed_frequency": (
                observed_frequency
            ),
            "calibration_error": (
                calibration_error
            ),
            "absolute_calibration_error": (
                abs(calibration_error)
            ),
        })

    return rows


# ============================================================
# 2. 80% INTERVAL COVERAGE + WIDTH
# ============================================================

def calculate_interval_80(
    Y,
    Q,
    levels,
):
    """
    Nominal 80% interval:

        [q_0.1, q_0.9]

    Perfect empirical coverage would be ~0.80.
    """

    q10_idx = get_quantile_index(
        levels,
        0.1,
    )

    q90_idx = get_quantile_index(
        levels,
        0.9,
    )

    lower = Q[:, :, q10_idx]
    upper = Q[:, :, q90_idx]

    coverage = float(
        np.mean(
            (Y >= lower)
            & (Y <= upper)
        )
    )

    width = float(
        np.mean(
            upper - lower
        )
    )

    return {
        "coverage_80": coverage,
        "coverage_error": (
            coverage - 0.80
        ),
        "absolute_coverage_error": (
            abs(coverage - 0.80)
        ),
        "mean_interval_width": width,
    }


# ============================================================
# 3. DOWNSIDE EXCEEDANCE AT h = 1, 5, 20
# ============================================================

def calculate_downside_exceedance(
    Y,
    Q,
    levels,
):
    """
    Frequency with which the realized return falls
    below the predicted 10% quantile.

    For a calibrated q_0.1 forecast this should be
    approximately 0.10.
    """

    q10_idx = get_quantile_index(
        levels,
        0.1,
    )

    q10 = Q[:, :, q10_idx]

    rows = []

    for h in SELECTED_HORIZONS:

        idx = h - 1

        observed_frequency = float(
            np.mean(
                Y[:, idx]
                <= q10[:, idx]
            )
        )

        rows.append({
            "horizon": h,
            "nominal_probability": 0.1,
            "observed_frequency": (
                observed_frequency
            ),
            "calibration_error": (
                observed_frequency
                - 0.1
            ),
            "absolute_calibration_error": (
                abs(
                    observed_frequency
                    - 0.1
                )
            ),
        })

    return rows


# ============================================================
# RUN ALL DAILY DATASETS
# ============================================================

reliability_rows = []
interval_rows = []
downside_rows = []


for model_name in MODEL_DIRS:

    print()
    print("=" * 70)
    print(f"MODEL: {model_name}")
    print("=" * 70)

    for asset_class, asset_name in ASSET_ORDER.items():

        test_asset_dir = (
            DATA_DIR
            / asset_class
        )

        test_files = sorted(
            test_asset_dir.glob(
                "test_*.csv"
            )
        )

        for test_path in test_files:

            try:

                dataset_name = (
                    test_path.stem
                    .removeprefix("test_")
                )

                # --------------------------------------------
                # Actual test targets
                # --------------------------------------------

                Y = load_test_y(
                    test_path
                )

                # --------------------------------------------
                # Saved model quantiles
                # --------------------------------------------

                quantile_path = (
                    find_quantile_file(
                        model_name=model_name,
                        asset_class=asset_class,
                        dataset_name=dataset_name,
                    )
                )

                Q, levels = (
                    load_quantiles(
                        quantile_path
                    )
                )

                validate_quantiles(
                    Y=Y,
                    Q=Q,
                    levels=levels,
                )

                # --------------------------------------------
                # Reliability
                # --------------------------------------------

                reliability = (
                    calculate_reliability(
                        Y=Y,
                        Q=Q,
                        levels=levels,
                    )
                )

                for row in reliability:

                    reliability_rows.append({
                        "model": model_name,
                        "asset_class": asset_name,
                        "dataset": dataset_name,
                        **row,
                    })

                # --------------------------------------------
                # 80% interval
                # --------------------------------------------

                interval = (
                    calculate_interval_80(
                        Y=Y,
                        Q=Q,
                        levels=levels,
                    )
                )

                interval_rows.append({
                    "model": model_name,
                    "asset_class": asset_name,
                    "dataset": dataset_name,
                    **interval,
                })

                # --------------------------------------------
                # Downside exceedance
                # --------------------------------------------

                downside = (
                    calculate_downside_exceedance(
                        Y=Y,
                        Q=Q,
                        levels=levels,
                    )
                )

                for row in downside:

                    downside_rows.append({
                        "model": model_name,
                        "asset_class": asset_name,
                        "dataset": dataset_name,
                        **row,
                    })

                print(
                    f"  {dataset_name}: done"
                )

            except Exception as exc:

                print(
                    f"  ERROR {test_path.name}: "
                    f"{exc}"
                )


# ============================================================
# DATAFRAMES
# ============================================================

reliability_df = pd.DataFrame(
    reliability_rows
)

interval_df = pd.DataFrame(
    interval_rows
)

downside_df = pd.DataFrame(
    downside_rows
)


# ============================================================
# MODEL-LEVEL SUMMARY
#
# Equal-series weighting:
# each dataset contributes equally.
# ============================================================

reliability_summary = (
    reliability_df
    .groupby(
        [
            "model",
            "quantile_level",
        ],
        as_index=False,
    )
    .agg(
        mean_observed_frequency=(
            "observed_frequency",
            "mean",
        ),
        mean_calibration_error=(
            "calibration_error",
            "mean",
        ),
        mean_absolute_calibration_error=(
            "absolute_calibration_error",
            "mean",
        ),
        n_datasets=(
            "dataset",
            "count",
        ),
    )
)


interval_summary = (
    interval_df
    .groupby(
        "model",
        as_index=False,
    )
    .agg(
        mean_coverage_80=(
            "coverage_80",
            "mean",
        ),
        mean_absolute_coverage_error=(
            "absolute_coverage_error",
            "mean",
        ),
        median_coverage_80=(
            "coverage_80",
            "median",
        ),
        mean_interval_width=(
            "mean_interval_width",
            "mean",
        ),
        median_interval_width=(
            "mean_interval_width",
            "median",
        ),
        n_datasets=(
            "dataset",
            "count",
        ),
    )
)


downside_summary = (
    downside_df
    .groupby(
        [
            "model",
            "horizon",
        ],
        as_index=False,
    )
    .agg(
        mean_observed_frequency=(
            "observed_frequency",
            "mean",
        ),
        mean_calibration_error=(
            "calibration_error",
            "mean",
        ),
        mean_absolute_calibration_error=(
            "absolute_calibration_error",
            "mean",
        ),
        n_datasets=(
            "dataset",
            "count",
        ),
    )
)


# ============================================================
# ASSET-CLASS SUMMARY
# ============================================================

reliability_asset_summary = (
    reliability_df
    .groupby(
        [
            "model",
            "asset_class",
            "quantile_level",
        ],
        as_index=False,
    )
    .agg(
        mean_observed_frequency=(
            "observed_frequency",
            "mean",
        ),
        mean_absolute_calibration_error=(
            "absolute_calibration_error",
            "mean",
        ),
        n_datasets=(
            "dataset",
            "count",
        ),
    )
)


interval_asset_summary = (
    interval_df
    .groupby(
        [
            "model",
            "asset_class",
        ],
        as_index=False,
    )
    .agg(
        mean_coverage_80=(
            "coverage_80",
            "mean",
        ),
        mean_absolute_coverage_error=(
            "absolute_coverage_error",
            "mean",
        ),
        mean_interval_width=(
            "mean_interval_width",
            "mean",
        ),
        n_datasets=(
            "dataset",
            "count",
        ),
    )
)


downside_asset_summary = (
    downside_df
    .groupby(
        [
            "model",
            "asset_class",
            "horizon",
        ],
        as_index=False,
    )
    .agg(
        mean_observed_frequency=(
            "observed_frequency",
            "mean",
        ),
        mean_absolute_calibration_error=(
            "absolute_calibration_error",
            "mean",
        ),
        n_datasets=(
            "dataset",
            "count",
        ),
    )
)


# ============================================================
# SAVE
# ============================================================

reliability_df.to_csv(
    BASE_DIR
    / "calibration_reliability_weekly.csv",
    index=False,
)

interval_df.to_csv(
    BASE_DIR
    / "calibration_interval80_weekly.csv",
    index=False,
)

downside_df.to_csv(
    BASE_DIR
    / "calibration_downside_horizons_weekly.csv",
    index=False,
)


reliability_summary.to_csv(
    BASE_DIR
    / "calibration_reliability_summary_weekly.csv",
    index=False,
)

interval_summary.to_csv(
    BASE_DIR
    / "calibration_interval80_summary_weekly.csv",
    index=False,
)

downside_summary.to_csv(
    BASE_DIR
    / "calibration_downside_summary_weekly.csv",
    index=False,
)


reliability_asset_summary.to_csv(
    BASE_DIR
    / "calibration_reliability_asset_class_weekly.csv",
    index=False,
)

interval_asset_summary.to_csv(
    BASE_DIR
    / "calibration_interval80_asset_class_weekly.csv",
    index=False,
)

downside_asset_summary.to_csv(
    BASE_DIR
    / "calibration_downside_asset_class_weekly.csv",
    index=False,
)


print()
print("=" * 70)
print("SAVED WEEKLY CALIBRATION DIAGNOSTICS")
print("=" * 70)

# Summary

In [ ]:
from pathlib import Path
import pandas as pd


# ============================================================
# CONFIG
# ============================================================

ROOT = Path(
    r"."
    r"\BForTFinProbability\BForTFin_results"
    r"\standardised"
)

FREQUENCIES = {
    "Daily": ROOT / "output_daily",
    "Weekly": ROOT / "output_weekly",
}


# ============================================================
# BUILD ONE FREQUENCY SUMMARY
# ============================================================

def build_calibration_summary(
    base_dir,
    frequency,
):

    # --------------------------------------------------------
    # Load existing summary files
    # --------------------------------------------------------

    reliability = pd.read_csv(
        base_dir
        / f"calibration_reliability_summary_{frequency.lower()}.csv"
    )

    interval = pd.read_csv(
        base_dir
        / f"calibration_interval80_summary_{frequency.lower()}.csv"
    )

    downside = pd.read_csv(
        base_dir
        / f"calibration_downside_summary_{frequency.lower()}.csv"
    )


    # ========================================================
    # 1. OVERALL RELIABILITY ERROR
    #
    # reliability currently has one row per:
    # model × quantile level
    #
    # Average absolute calibration error across q=0.1,...,0.9
    # ========================================================

    reliability_overall = (
        reliability
        .groupby(
            "model",
            as_index=False,
        )
        .agg(
            mean_abs_reliability_error=(
                "mean_absolute_calibration_error",
                "mean",
            ),
            mean_signed_reliability_error=(
                "mean_calibration_error",
                "mean",
            ),
        )
    )


    # ========================================================
    # 2. INTERVAL RESULTS
    # ========================================================

    interval_keep = interval[
        [
            "model",
            "mean_coverage_80",
            "mean_absolute_coverage_error",
            "mean_interval_width",
        ]
    ].copy()


    # ========================================================
    # 3. DOWNSIDE q0.1 EXCEEDANCE
    #
    # Convert:
    #
    # model | horizon | observed freq
    #
    # into:
    #
    # model | q10_h1 | q10_h5 | q10_h20
    # ========================================================

    downside_keep = downside[
        downside["horizon"].isin(
            [1, 5, 20]
        )
    ].copy()

    downside_pivot = (
        downside_keep
        .pivot(
            index="model",
            columns="horizon",
            values="mean_observed_frequency",
        )
        .reset_index()
    )

    downside_pivot = downside_pivot.rename(
        columns={
            1: "q10_exceedance_h1",
            5: "q10_exceedance_h5",
            20: "q10_exceedance_h20",
        }
    )


    # ========================================================
    # MERGE EVERYTHING
    # ========================================================

    summary = (
        reliability_overall
        .merge(
            interval_keep,
            on="model",
            how="outer",
        )
        .merge(
            downside_pivot,
            on="model",
            how="outer",
        )
    )

    summary.insert(
        0,
        "frequency",
        frequency,
    )


    # ========================================================
    # ADD SIMPLE INTERPRETATION METRICS
    # ========================================================

    # Mean absolute q0.1 calibration error across h=1,5,20

    summary["mean_abs_q10_error_h1_5_20"] = (
        (
            (summary["q10_exceedance_h1"] - 0.10).abs()
            + (summary["q10_exceedance_h5"] - 0.10).abs()
            + (summary["q10_exceedance_h20"] - 0.10).abs()
        )
        / 3
    )


    # ========================================================
    # SORT
    #
    # Lowest reliability error first
    # ========================================================

    summary = summary.sort_values(
        "mean_abs_reliability_error",
        ascending=True,
    ).reset_index(drop=True)

    return summary


# ============================================================
# DAILY + WEEKLY
# ============================================================

all_summaries = []

for frequency, base_dir in FREQUENCIES.items():

    summary = build_calibration_summary(
        base_dir=base_dir,
        frequency=frequency,
    )

    # Save frequency-specific table

    output_path = (
        base_dir
        / f"calibration_compact_summary_{frequency.lower()}.csv"
    )

    summary.to_csv(
        output_path,
        index=False,
    )

    print()
    print("=" * 80)
    print(f"{frequency.upper()} CALIBRATION")
    print("=" * 80)

    print(
        summary.round(4).to_string(
            index=False
        )
    )

    print()
    print(
        f"Saved to: {output_path}"
    )

    all_summaries.append(
        summary
    )


# ============================================================
# COMBINED DAILY + WEEKLY TABLE
# ============================================================

combined = pd.concat(
    all_summaries,
    ignore_index=True,
)

combined_path = (
    ROOT
    / "calibration_compact_summary_daily_weekly.csv"
)

combined.to_csv(
    combined_path,
    index=False,
)

print()
print("=" * 80)
print("COMBINED TABLE SAVED")
print("=" * 80)
print(combined_path)

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt


ROOT = Path(
    r"."
    r"\BForTFinProbability\BForTFin_results"
    r"\standardised"
)

FREQUENCIES = {
    "Daily": ROOT / "output_daily",
    "Weekly": ROOT / "output_weekly",
}


for frequency, base_dir in FREQUENCIES.items():

    df = pd.read_csv(
        base_dir
        / f"calibration_reliability_summary_{frequency.lower()}.csv"
    )

    fig, ax = plt.subplots(
        figsize=(8, 6)
    )

    # --------------------------------------------------------
    # Perfect calibration line
    # --------------------------------------------------------

    ax.plot(
        [0.1, 0.9],
        [0.1, 0.9],
        linestyle="--",
        linewidth=1.5,
        label="Perfect calibration",
    )


    # --------------------------------------------------------
    # One reliability curve per model
    # --------------------------------------------------------

    for model, group in df.groupby("model"):

        group = group.sort_values(
            "quantile_level"
        )

        ax.plot(
            group["quantile_level"],
            group["mean_observed_frequency"],
            marker="o",
            linewidth=1.5,
            label=model,
        )


    # --------------------------------------------------------
    # Formatting
    # --------------------------------------------------------

    ax.set_xlabel(
        "Nominal Quantile Level",
        fontsize=12,
    )

    ax.set_ylabel(
        "Observed Frequency Below Predicted Quantile",
        fontsize=12,
    )

    ax.set_title(
        f"{frequency} Quantile Reliability",
        fontsize=14,
    )

    ax.set_xlim(
        0.08,
        0.92,
    )

    ax.set_ylim(
        0.0,
        1.0,
    )

    ax.grid(
        alpha=0.25
    )

    ax.legend(
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        fontsize=9,
    )

    fig.tight_layout()

    output_path = (
        base_dir
        / f"calibration_reliability_plot_{frequency.lower()}.png"
    )

    fig.savefig(
        output_path,
        dpi=300,
        bbox_inches="tight",
    )

    plt.show()

    print(
        f"Saved to: {output_path}"
    )

# Asset class summary

In [ ]:
from pathlib import Path
import pandas as pd


# ============================================================
# CONFIG
# ============================================================

ROOT = Path(
    r"."
    r"\BForTFinProbability\BForTFin_results"
    r"\standardised"
)

FREQUENCIES = {
    "Daily": ROOT / "output_daily",
    "Weekly": ROOT / "output_weekly",
}


# ============================================================
# BUILD ASSET-CLASS CALIBRATION TABLE
# ============================================================

def build_asset_class_calibration_table(
    base_dir,
    frequency,
):

    freq_lower = frequency.lower()

    # --------------------------------------------------------
    # Load asset-class summaries
    # --------------------------------------------------------

    reliability = pd.read_csv(
        base_dir
        / f"calibration_reliability_asset_class_{freq_lower}.csv"
    )

    interval = pd.read_csv(
        base_dir
        / f"calibration_interval80_asset_class_{freq_lower}.csv"
    )

    downside = pd.read_csv(
        base_dir
        / f"calibration_downside_asset_class_{freq_lower}.csv"
    )


    # ========================================================
    # 1. OVERALL RELIABILITY MAE BY MODEL × ASSET CLASS
    #
    # reliability currently has one row per:
    # model × asset_class × quantile_level
    #
    # Average over q = 0.1,...,0.9
    # ========================================================

    reliability_overall = (
        reliability
        .groupby(
            [
                "model",
                "asset_class",
            ],
            as_index=False,
        )
        .agg(
            reliability_mae=(
                "mean_absolute_calibration_error",
                "mean",
            ),
        )
    )


    # ========================================================
    # 2. 80% INTERVAL RESULTS
    # ========================================================

    interval_keep = interval[
        [
            "model",
            "asset_class",
            "mean_coverage_80",
            "mean_absolute_coverage_error",
            "mean_interval_width",
        ]
    ].copy()


    # ========================================================
    # 3. q0.1 EXCEEDANCE AT h = 1, 5, 20
    # ========================================================

    downside_keep = downside[
        downside["horizon"].isin(
            [1, 5, 20]
        )
    ].copy()

    downside_pivot = (
        downside_keep
        .pivot_table(
            index=[
                "model",
                "asset_class",
            ],
            columns="horizon",
            values="mean_observed_frequency",
            aggfunc="first",
        )
        .reset_index()
    )

    downside_pivot = downside_pivot.rename(
        columns={
            1: "q10_exceedance_h1",
            5: "q10_exceedance_h5",
            20: "q10_exceedance_h20",
        }
    )


    # ========================================================
    # 4. MERGE
    # ========================================================

    table = (
        reliability_overall
        .merge(
            interval_keep,
            on=[
                "model",
                "asset_class",
            ],
            how="outer",
        )
        .merge(
            downside_pivot,
            on=[
                "model",
                "asset_class",
            ],
            how="outer",
        )
    )

    table.insert(
        0,
        "frequency",
        frequency,
    )


    # ========================================================
    # 5. OPTIONAL:
    # MEAN ABSOLUTE q0.1 ERROR ACROSS h=1,5,20
    # ========================================================

    table["q10_mae_h1_5_20"] = (
        (
            (table["q10_exceedance_h1"] - 0.10).abs()
            + (table["q10_exceedance_h5"] - 0.10).abs()
            + (table["q10_exceedance_h20"] - 0.10).abs()
        )
        / 3
    )


    # ========================================================
    # 6. SORT
    # ========================================================

    asset_order = [
        "FX",
        "EM FX",
        "MSCI",
        "Rates",
        "REIT",
        "Equities",
        "ETF",
        "Crypto",
    ]

    table["asset_class"] = pd.Categorical(
        table["asset_class"],
        categories=asset_order,
        ordered=True,
    )

    table = table.sort_values(
        [
            "asset_class",
            "reliability_mae",
            "model",
        ]
    ).reset_index(drop=True)

    display_table = table[
        [
            "frequency",
            "model",
            "asset_class",
            "reliability_mae",
            "mean_coverage_80",
            "q10_exceedance_h1",
            "q10_exceedance_h5",
            "q10_exceedance_h20",
        ]
    ].copy()
    
    display_table = display_table.rename(
        columns={
            "frequency": "Frequency",
            "model": "Model",
            "asset_class": "Asset Class",
            "reliability_mae": "Reliability MAE",
            "mean_coverage_80": "80% Coverage",
            "q10_exceedance_h1": "q0.1 Exceedance h=1",
            "q10_exceedance_h5": "q0.1 Exceedance h=5",
            "q10_exceedance_h20": "q0.1 Exceedance h=20",
        }
    )
    
    print(
        display_table
        .round(3)
        .to_string(index=False)
    )


    return table


# ============================================================
# RUN DAILY + WEEKLY
# ============================================================

all_tables = []

for frequency, base_dir in FREQUENCIES.items():

    table = build_asset_class_calibration_table(
        base_dir=base_dir,
        frequency=frequency,
    )

    output_path = (
        base_dir
        / f"calibration_asset_class_table_{frequency.lower()}.csv"
    )

    table.to_csv(
        output_path,
        index=False,
    )

    print()
    print("=" * 100)
    print(
        f"{frequency.upper()} ASSET-CLASS CALIBRATION"
    )
    print("=" * 100)

    print(
        table.round(4).to_string(
            index=False
        )
    )

    print()
    print(f"Saved to: {output_path}")

    all_tables.append(
        table
    )


# ============================================================
# COMBINED DAILY + WEEKLY
# ============================================================

combined = pd.concat(
    all_tables,
    ignore_index=True,
)

combined_path = (
    ROOT
    / "calibration_asset_class_table_daily_weekly.csv"
)

combined.to_csv(
    combined_path,
    index=False,
)

print()
print("=" * 100)
print("COMBINED ASSET-CLASS TABLE SAVED")
print("=" * 100)
print(combined_path)

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np


# ============================================================
# CONFIG
# ============================================================

ROOT = Path(
    r"."
    r"\BForTFinProbability\BForTFin_results"
    r"\standardised"
)

INPUT_PATH = (
    ROOT
    / "calibration_asset_class_table_daily_weekly.csv"
)

OUTPUT_PATH = (
    ROOT
    / "calibration_asset_class_analysis.csv"
)


# ============================================================
# LOAD
# ============================================================

df = pd.read_csv(
    INPUT_PATH
)


# ============================================================
# DERIVED ERRORS
# ============================================================

# 80% interval coverage:
# smaller error from 0.80 = better
df["coverage80_error"] = (
    df["mean_coverage_80"]
    - 0.80
).abs()


# q0.1 exceedance:
# smaller error from 0.10 = better

df["q10_error_h1"] = (
    df["q10_exceedance_h1"]
    - 0.10
).abs()

df["q10_error_h5"] = (
    df["q10_exceedance_h5"]
    - 0.10
).abs()

df["q10_error_h20"] = (
    df["q10_exceedance_h20"]
    - 0.10
).abs()


df["q10_mean_error"] = (
    df[
        [
            "q10_error_h1",
            "q10_error_h5",
            "q10_error_h20",
        ]
    ]
    .mean(axis=1)
)


# ============================================================
# RANK WITHIN EACH FREQUENCY × ASSET CLASS
# ============================================================

group_cols = [
    "frequency",
    "asset_class",
]


# Lower = better for all of these
df["rank_reliability"] = (
    df
    .groupby(group_cols)[
        "reliability_mae"
    ]
    .rank(
        method="average",
        ascending=True,
    )
)


df["rank_coverage80"] = (
    df
    .groupby(group_cols)[
        "coverage80_error"
    ]
    .rank(
        method="average",
        ascending=True,
    )
)


df["rank_q10"] = (
    df
    .groupby(group_cols)[
        "q10_mean_error"
    ]
    .rank(
        method="average",
        ascending=True,
    )
)


# ============================================================
# OVERALL CALIBRATION RANK
#
# Equal weight to:
# - reliability
# - 80% interval coverage
# - q0.1 exceedance
# ============================================================

df["mean_calibration_rank"] = (
    df[
        [
            "rank_reliability",
            "rank_coverage80",
            "rank_q10",
        ]
    ]
    .mean(axis=1)
)


# ============================================================
# BEST MODEL PER ASSET CLASS
# ============================================================

best_reliability = (
    df.loc[
        df.groupby(group_cols)[
            "reliability_mae"
        ].idxmin()
    ][
        [
            "frequency",
            "asset_class",
            "model",
            "reliability_mae",
        ]
    ]
    .rename(
        columns={
            "model": "best_reliability_model",
            "reliability_mae": "best_reliability_mae",
        }
    )
)


best_coverage = (
    df.loc[
        df.groupby(group_cols)[
            "coverage80_error"
        ].idxmin()
    ][
        [
            "frequency",
            "asset_class",
            "model",
            "mean_coverage_80",
            "coverage80_error",
        ]
    ]
    .rename(
        columns={
            "model": "best_coverage_model",
            "mean_coverage_80": "best_coverage80",
            "coverage80_error": "best_coverage80_error",
        }
    )
)


best_q10 = (
    df.loc[
        df.groupby(group_cols)[
            "q10_mean_error"
        ].idxmin()
    ][
        [
            "frequency",
            "asset_class",
            "model",
            "q10_mean_error",
        ]
    ]
    .rename(
        columns={
            "model": "best_q10_model",
            "q10_mean_error": "best_q10_mean_error",
        }
    )
)


best_overall = (
    df.loc[
        df.groupby(group_cols)[
            "mean_calibration_rank"
        ].idxmin()
    ][
        [
            "frequency",
            "asset_class",
            "model",
            "mean_calibration_rank",
        ]
    ]
    .rename(
        columns={
            "model": "best_overall_calibration_model",
        }
    )
)


# ============================================================
# COMBINE BEST-MODEL SUMMARY
# ============================================================

best_summary = (
    best_reliability
    .merge(
        best_coverage,
        on=group_cols,
    )
    .merge(
        best_q10,
        on=group_cols,
    )
    .merge(
        best_overall,
        on=group_cols,
    )
)


# ============================================================
# MODEL CONSISTENCY ACROSS ASSET CLASSES
#
# Average rank across all asset classes
# separately for Daily / Weekly
# ============================================================

model_consistency = (
    df
    .groupby(
        [
            "frequency",
            "model",
        ],
        as_index=False,
    )
    .agg(
        mean_reliability_rank=(
            "rank_reliability",
            "mean",
        ),
        mean_coverage_rank=(
            "rank_coverage80",
            "mean",
        ),
        mean_q10_rank=(
            "rank_q10",
            "mean",
        ),
        mean_overall_calibration_rank=(
            "mean_calibration_rank",
            "mean",
        ),
        mean_reliability_mae=(
            "reliability_mae",
            "mean",
        ),
        mean_coverage80_error=(
            "coverage80_error",
            "mean",
        ),
        mean_q10_error=(
            "q10_mean_error",
            "mean",
        ),
    )
)


model_consistency = (
    model_consistency
    .sort_values(
        [
            "frequency",
            "mean_overall_calibration_rank",
        ]
    )
    .reset_index(drop=True)
)


# ============================================================
# COUNT "WINS" ACROSS ASSET CLASSES
# ============================================================

overall_wins = (
    best_overall
    .groupby(
        [
            "frequency",
            "best_overall_calibration_model",
        ]
    )
    .size()
    .reset_index(
        name="n_asset_class_wins"
    )
    .rename(
        columns={
            "best_overall_calibration_model": "model"
        }
    )
)


reliability_wins = (
    best_reliability
    .groupby(
        [
            "frequency",
            "best_reliability_model",
        ]
    )
    .size()
    .reset_index(
        name="n_reliability_wins"
    )
    .rename(
        columns={
            "best_reliability_model": "model"
        }
    )
)


# ============================================================
# SAVE DETAILED RANKED TABLE
# ============================================================

ranked_output = df[
    [
        "frequency",
        "asset_class",
        "model",
        "reliability_mae",
        "mean_coverage_80",
        "coverage80_error",
        "q10_exceedance_h1",
        "q10_exceedance_h5",
        "q10_exceedance_h20",
        "q10_mean_error",
        "rank_reliability",
        "rank_coverage80",
        "rank_q10",
        "mean_calibration_rank",
    ]
].copy()


ranked_output = ranked_output.sort_values(
    [
        "frequency",
        "asset_class",
        "mean_calibration_rank",
    ]
)


ranked_output.to_csv(
    OUTPUT_PATH,
    index=False,
)


# ============================================================
# SAVE OTHER SUMMARIES
# ============================================================

best_summary.to_csv(
    ROOT
    / "calibration_best_model_by_asset_class.csv",
    index=False,
)

model_consistency.to_csv(
    ROOT
    / "calibration_model_consistency_by_frequency.csv",
    index=False,
)

overall_wins.to_csv(
    ROOT
    / "calibration_asset_class_wins.csv",
    index=False,
)

reliability_wins.to_csv(
    ROOT
    / "calibration_reliability_wins.csv",
    index=False,
)


# ============================================================
# PRINT
# ============================================================

print()
print("=" * 100)
print("BEST MODELS BY ASSET CLASS")
print("=" * 100)

print(
    best_summary
    .round(4)
    .to_string(index=False)
)


print()
print("=" * 100)
print("MODEL CONSISTENCY ACROSS ASSET CLASSES")
print("=" * 100)

print(
    model_consistency
    .round(4)
    .to_string(index=False)
)


print()
print("=" * 100)
print("OVERALL CALIBRATION WINS")
print("=" * 100)

print(
    overall_wins.to_string(
        index=False
    )
)